In [1]:
import json
import re
import string
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

### Load Path

In [5]:
path = Path('../datasets/Entity Recognition in Resumes.json')

Entity_labels = [
    'Name',
    'Designation',
    'Companies worked at', 
    'Location',
    'Email Address', 
    'College Name', 
    'Degree', 
    'Graduation Year',
    'Skills', 
    'Years of Experience'
]

Non_lemma_label ={
    'Name', 
    'Email Address', 
    'Graduation Year', 
    'Location',
    'College Name', 
    'Companies worked at'
}

print(f"Entity types: {len(Entity_labels)}")

Entity types: 10


### Data Parsing

In [20]:
def load_jsonl(path: Path) -> list[dict]:
    """Load all records from a JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def parse_record(record: dict) -> tuple[str, list[tuple]]:
    """
    Extract (content, spans) from a single record.

    Returns
    -------
    content : str
        Raw resume text.
    spans : list of (start, end, label)
        Character-level offsets where end is EXCLUSIVE (Python slice convention).
        Sorted by start position. UNKNOWN labels are dropped.
    """
    content = record['content']
    spans = []
    for ann in record.get('annotation', []):
        label = ann['label'][0] if ann['label'] else 'UNKNOWN'
        if label == 'UNKNOWN':
            continue
        for pt in ann['points']:
            start = pt['start']
            end   = pt['end'] + 1          # convert inclusive -> exclusive
            # Sanity check: stored text must match the slice
            if content[start:end] != pt['text']:
                continue
            spans.append((start, end, label))

    # Sort by start; resolve any end ties by longest span first
    spans.sort(key=lambda s: (s[0], -s[1]))
    return content, spans


# Load & parse all records
raw_records = load_jsonl(path)
parsed = [parse_record(r) for r in raw_records]

print(f'Records loaded  : {len(parsed)}')
print(f'Total spans     : {sum(len(s) for _, s in parsed)}')
print()

# Inspect first record
content0, spans0 = parsed[0]
print(f"=== Resume 0 (first 200 chars) ===")
print(content0[:200])
print(f"\n=== Spans (first 10) ===")
for start, end, label in spans0[:10]:
    print(f"  [{start:5d}:{end:5d}]  {label:<25}  '{content0[start:end][:60]!s}'")

Records loaded  : 220
Total spans     : 3333

=== Resume 0 (first 200 chars) ===
Abhishek Jha
Application Development Associate - Accenture

Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a

• To work for an organization which provides me the o

=== Spans (first 10) ===
  [    0:   12]  Name                       'Abhishek Jha'
  [   13:   46]  Designation                'Application Development Associate'
  [   49:   58]  Companies worked at        'Accenture'
  [   60:   69]  Location                   'Bengaluru'
  [   95:  146]  Email Address              'Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a
'
  [  372:  405]  Designation                'Application Development Associate'
  [  407:  416]  Companies worked at        'Accenture'
  [  727:  770]  Designation                'B.E in Information science and engineering
'
  [  771:  814]  College Name               'B.v.b college of engineering and technology'
  [  856:  861]  Graduation